# FinDER × FIBO: how each module changes KG construction quality

FIBO ships as a stack of modules — Business Entities (**BE**), Financial Business and Commerce (**FBC**), Securities (**SEC**), Foundations (**FND**), Indices and Indicators (**IND**) — and you don't always need all of them. This notebook quantifies how each module changes the quality of a knowledge graph extracted from FinDER (SEC 10-K) text.

**Sweep:** five ontology configurations on the same FinDER documents:

1. `none`         — generic baseline (just `Entity` / `RELATED_TO`)
2. `be`           — Business Entities only
3. `be+fbc`       — + Financial Business and Commerce
4. `be+fbc+sec`   — + Securities
5. `full`         — BE + FBC + SEC + FND + IND

**Metrics per config:**
- *Volume* — node/edge count, distinct labels, distinct rel types
- *Validity* — soft SHACL-style checks (required props, label allowlist)
- *Coverage* — expected named entities recovered from each FinDER document
- *Answer quality* — contains-match rate against FinDER expected answers

Module YAML slices live under `examples/finder/datasets/fibo_modules/`; the composer is `compose.py`. Tweak/extend the slices to model your own FIBO subset.

## Setup

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "seocho").is_dir() and (p / "examples").is_dir():
            return p
        p = p.parent
    return Path.cwd()

ROOT = _find_repo_root()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

LLM_SPEC = os.environ.get("SEOCHO_LLM", "openai/gpt-4o-mini")
LLM_PROVIDER, LLM_MODEL = (LLM_SPEC.split("/", 1) + [""])[:2]
if not LLM_MODEL:
    LLM_PROVIDER, LLM_MODEL = "openai", LLM_SPEC

PROVIDER_KEY = {
    "openai": "OPENAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "kimi": "MOONSHOT_API_KEY",
    "grok": "XAI_API_KEY",
    "qwen": "DASHSCOPE_API_KEY",
}.get(LLM_PROVIDER, "OPENAI_API_KEY")

if not os.environ.get(PROVIDER_KEY):
    raise RuntimeError(f"{PROVIDER_KEY} required for SEOCHO_LLM={LLM_SPEC}")

FINDER_PATH = os.environ.get(
    "FINDER_PATH",
    str(ROOT / "examples" / "finder" / "datasets" / "finder_tutorial_subset.json"),
)
WORKDIR = ROOT / ".seocho" / "finder_fibo_impact"
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"FinDER source: {FINDER_PATH}")
print(f"Working dir:   {WORKDIR}")
print(f"LLM:           {LLM_PROVIDER}/{LLM_MODEL}")

In [2]:
from seocho.benchmarking import load_finder_cases

cases = load_finder_cases(FINDER_PATH)
print(f"Loaded {len(cases)} FinDER cases")

Loaded 10 FinDER cases


## Define the five configurations

In [3]:
from examples.finder.datasets.fibo_modules.compose import compose_modules

CONFIGS = {
    "none":          [],
    "be":            ["be"],
    "be+fbc":        ["be", "fbc"],
    "be+fbc+sec":    ["be", "fbc", "sec"],
    "full":          ["be", "fbc", "sec", "fnd", "ind"],
}

for label, modules in CONFIGS.items():
    onto = compose_modules(modules)
    print(f"  {label:14s} nodes={len(onto.nodes):2d} rels={len(onto.relationships):2d}")

  none           nodes= 1 rels= 1
  be             nodes= 2 rels= 2
  be+fbc         nodes= 5 rels= 5
  be+fbc+sec     nodes= 8 rels= 8
  full           nodes=14 rels=14


## Pick the entities we expect to find

For coverage scoring we need a per-document gold list of named entities. For the synthetic subset this is hand-curated; for a real FinDER run, replace with NER output or a curated answer-key.

In [4]:
EXPECTED_ENTITIES = {
    "finder_tut_001": ["Apple Inc.", "Cupertino", "AAPL"],
    "finder_tut_002": ["Microsoft Corporation", "Azure", "Intelligent Cloud"],
    "finder_tut_003": ["NVIDIA Corporation"],
    "finder_tut_004": ["Alphabet Inc.", "Sundar Pichai", "John L. Hennessy", "Frances H. Arnold", "L. John Doerr"],
    "finder_tut_005": ["Meta Platforms, Inc.", "Federal Trade Commission", "WhatsApp", "Instagram"],
    "finder_tut_006": ["Tesla, Inc."],
    "finder_tut_007": ["JPMorgan Chase & Co."],
    "finder_tut_008": ["Berkshire Hathaway Inc.", "GEICO", "General Re"],
    "finder_tut_009": ["Amazon.com, Inc.", "Andy Jassy", "Amazon Web Services"],
    "finder_tut_010": ["Berkshire Hathaway", "Apple Inc.", "Bank of America", "American Express", "Coca-Cola"],
}

## Sweep — extract once per config, capture nodes and relationships

In [ ]:
from seocho import Seocho
from seocho.index.pipeline import IndexingPipeline
from seocho.store.graph import LadybugGraphStore
from seocho.store.llm import create_llm_backend

def run_config(label: str, modules: list[str]):
    onto = compose_modules(modules)
    db_path = WORKDIR / f"{label.replace('+', '_')}.lbug"
    # Clean up the .lbug file *and* its WAL/SHM siblings so re-runs
    # don't trip LadybugDB's "database ID mismatch" check.
    for sibling in db_path.parent.glob(db_path.name + "*"):
        try:
            sibling.unlink()
        except FileNotFoundError:
            pass
    store = LadybugGraphStore(str(db_path))
    try:
        store.ensure_constraints(onto)
    except Exception:
        pass
    llm_backend = create_llm_backend(provider=LLM_PROVIDER, model=LLM_MODEL)

    nodes_by_case: dict[str, list] = {c.case_id: [] for c in cases}
    rels_by_case: dict[str, list] = {c.case_id: [] for c in cases}
    current_case = {"id": ""}
    def capture_extract(nodes, rels):
        cid = current_case["id"]
        if cid:
            nodes_by_case[cid].extend(nodes)
            rels_by_case[cid].extend(rels)
        return nodes, rels
    pipeline = IndexingPipeline(
        ontology=onto,
        graph_store=store,
        llm=llm_backend,
        workspace_id=f"finder_{label}",
        on_after_extract=capture_extract,
    )
    add_latency_ms: dict[str, float] = {}
    for case in cases:
        current_case["id"] = case.case_id
        t0 = time.perf_counter()
        try:
            pipeline.index(case.text, metadata={"case_id": case.case_id, "category": case.category})
        except Exception as exc:
            print(f"  index failure on {case.case_id}: {exc}")
        add_latency_ms[case.case_id] = (time.perf_counter() - t0) * 1000
    current_case["id"] = ""

    client = Seocho(
        ontology=onto,
        graph_store=store,
        llm=llm_backend,
        workspace_id=f"finder_{label}",
    )
    qa_results: list[dict] = []
    for case in cases:
        try:
            answer = client.ask(case.question)
        except Exception as exc:
            answer = f"<error: {type(exc).__name__}>"
        qa_results.append({
            "case_id": case.case_id,
            "question": case.question,
            "answer": answer,
            "expected": case.expected_answer,
        })
    return {
        "ontology": onto,
        "nodes_by_case": nodes_by_case,
        "rels_by_case": rels_by_case,
        "add_latency_ms": add_latency_ms,
        "qa_results": qa_results,
    }

results = {}
for label, modules in CONFIGS.items():
    print(f"== running config: {label} ==")
    results[label] = run_config(label, modules)
    n_nodes = sum(len(v) for v in results[label]["nodes_by_case"].values())
    n_rels = sum(len(v) for v in results[label]["rels_by_case"].values())
    print(f"   {n_nodes} nodes, {n_rels} rels")

## Compute metrics per config

In [ ]:
from examples.finder.lib.fibo_module_metrics import (
    entity_coverage,
    graph_volume,
    label_distribution,
    qa_correctness,
    shacl_violation_count,
)

metrics = {}
for label, res in results.items():
    all_nodes = [n for ns in res["nodes_by_case"].values() for n in ns]
    all_rels = [r for rs in res["rels_by_case"].values() for r in rs]
    coverage_per_case = []
    for case in cases:
        cov = entity_coverage(
            res["nodes_by_case"].get(case.case_id, []),
            EXPECTED_ENTITIES.get(case.case_id, []),
        )
        coverage_per_case.append(cov["coverage"])
    metrics[label] = {
        **graph_volume(all_nodes, all_rels),
        "label_distribution": label_distribution(all_nodes),
        "shacl": shacl_violation_count(all_nodes, all_rels, res["ontology"]),
        "avg_coverage": sum(coverage_per_case) / len(coverage_per_case),
        "qa": qa_correctness(res["qa_results"]),
        "avg_add_latency_ms": sum(res["add_latency_ms"].values()) / max(1, len(res["add_latency_ms"])),
    }

(WORKDIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
for label, m in metrics.items():
    print(f"\n== {label} ==")
    print(f"  nodes={m['node_count']}  rels={m['relationship_count']}  labels={m['distinct_labels']}  rel_types={m['distinct_rel_types']}")
    print(f"  coverage avg={m['avg_coverage']:.2f}  shacl_violations={m['shacl']['total']}")
    print(f"  contains_match={m['qa']['contains_match_rate']:.2f}  add_avg_ms={m['avg_add_latency_ms']:.0f}")

## Visualize impact

In [ ]:
import matplotlib.pyplot as plt

labels = list(metrics.keys())
coverage = [metrics[l]["avg_coverage"] for l in labels]
qa = [metrics[l]["qa"]["contains_match_rate"] for l in labels]
node_counts = [metrics[l]["node_count"] for l in labels]
rel_counts = [metrics[l]["relationship_count"] for l in labels]
shacl = [metrics[l]["shacl"]["total"] for l in labels]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].bar(labels, coverage)
axes[0].set_title("Avg entity coverage")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(labels, qa)
axes[1].set_title("FinDER contains-match rate")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=30)
axes[2].bar(labels, node_counts, label="nodes")
axes[2].bar(labels, rel_counts, label="rels", alpha=0.6)
axes[2].set_title("Graph size")
axes[2].legend()
axes[2].tick_params(axis='x', rotation=30)
axes[3].bar(labels, shacl)
axes[3].set_title("SHACL-style violations")
axes[3].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## What to read off the chart

- *Coverage* climbing as modules accumulate means each module is unlocking new types of entities the LLM can ground in your text. Plateaus point to redundant scope on this corpus.
- *contains-match rate* is the question-answer signal. Bigger ontologies don't always answer better — they can over-extract and confuse the deterministic Cypher generator.
- *SHACL violations* on the bigger configs reveals how strictly the LLM honors the schema. A non-trivial number on the `none` config is normal because the baseline only allows the `Entity` label.
- *Graph size* growing super-linearly with module count usually signals duplicate concepts across modules — fold the duplicates with `Ontology.merge` overrides if so.

The full per-config payload (including label distribution) is at `.seocho/finder_fibo_impact/metrics.json`. Cross-check that with `docs/SHACL_GRAPHRAG_PLAYBOOK.md` for SHACL-validation guidance once you're ready to wire `pyshacl` against `Ontology.to_shacl()`.

## Visualize the actual graph shape

The bar charts above summarize quality metrics. The graphs themselves look very different too — `none` produces a flat `Entity / RELATED_TO` graph, while `full` projects rich FIBO labels and typed relationships. Side-by-side:

In [ ]:
from examples.finder.lib.graph_viz import draw_lpg
import matplotlib.pyplot as plt

# Pick two configs to compare visually.
LEFT, RIGHT = "none", "full"

for label in (LEFT, RIGHT):
    res = results[label]
    nodes = [n for ns in res["nodes_by_case"].values() for n in ns]
    rels  = [r for rs in res["rels_by_case"].values() for r in rs]
    fig = draw_lpg(
        nodes, rels,
        title=f"{label}  —  {len(nodes)} nodes / {len(rels)} rels",
        max_nodes=80,
        figsize=(11, 7),
    )
    plt.show()